# STRAT-002 v5 - Deep Dive Investigation

Understanding the walk-forward results vs full backtest discrepancy.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/daily")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df.dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Full data from 2019
full_data = df[df.index >= '2019-01-01'].copy()

# Entry signals
cond = (full_data['sopr'] < 1) & (full_data['sopr_sth'] < 1) & (full_data['rl_zscore'] > 0.5)
entries = cond & ~cond.shift(1).fillna(False)

print(f"Total entry signals: {entries.sum()}")
print(f"\nEntry dates:")
for date in entries[entries].index:
    print(f"  {date.date()}: ${full_data.loc[date, 'price']:,.0f}")

In [ ]:
# Run full backtest
pf = vbt.Portfolio.from_signals(
    close=full_data['price'],
    entries=entries,
    exits=None,
    sl_stop=0.30,
    sl_trail=True,
    stop_exit_price='close',
    fees=0.001,
    init_cash=100000,
    freq='D'
)

print("FULL BACKTEST RESULTS")
print("="*60)
print(pf.stats())

In [ ]:
# Detailed trade log
print("\nDETAILED TRADE LOG")
print("="*100)
print(pf.trades.records_readable.to_string())

In [ ]:
# Check what buy & hold would be
bh_return = (full_data['price'].iloc[-1] / full_data['price'].iloc[0] - 1) * 100
strat_return = pf.total_return() * 100

print(f"\nCOMPARISON")
print("="*60)
print(f"Strategy Return: {strat_return:+,.0f}%")
print(f"Buy & Hold Return: {bh_return:+,.0f}%")
print(f"Difference: {strat_return - bh_return:+,.0f}%")
print(f"")
print(f"Strategy beats B&H: {'✅ YES' if strat_return > bh_return else '❌ NO'}")

In [ ]:
# Year by year - but tracking continuous equity
print("\nYEAR-BY-YEAR BREAKDOWN (from full backtest)")
print("="*80)

# Get equity curve
equity = pf.value()

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
print(f"{'Year':<8} {'Start $':>14} {'End $':>14} {'Return':>12} {'B&H':>12}")
print("-"*80)

for year in years:
    year_start = f'{year}-01-01'
    year_end = f'{year}-12-31'
    
    year_equity = equity[(equity.index >= year_start) & (equity.index <= year_end)]
    year_price = full_data[(full_data.index >= year_start) & (full_data.index <= year_end)]['price']
    
    if len(year_equity) > 0 and len(year_price) > 0:
        start_eq = year_equity.iloc[0]
        end_eq = year_equity.iloc[-1]
        strat_ret = (end_eq / start_eq - 1) * 100
        bh_ret = (year_price.iloc[-1] / year_price.iloc[0] - 1) * 100
        
        print(f"{year:<8} ${start_eq:>13,.0f} ${end_eq:>13,.0f} {strat_ret:>+11.0f}% {bh_ret:>+11.0f}%")

In [ ]:
# Plot equity curve
pf.plot().show()

In [ ]:
# Check if we're in a position now
print("\nCURRENT STATUS")
print("="*60)
print(f"Currently in position: {pf.position_mask().iloc[-1]}")
print(f"Current equity: ${pf.value().iloc[-1]:,.0f}")
print(f"Current BTC price: ${full_data['price'].iloc[-1]:,.0f}")